# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description
print("Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)


## 2. Data Overview
Review available record sets, fields, and their `@id`s for exploration. 

In [ ]:
# Display available record sets
if not dataset.record_sets:
    print("No record sets found in the Croissant schema at top level. Attempting to discover from schema content...")
    # Workaround for datasets with non-standard schema
    croissant_json = dataset._metadata_json  # internal, usually contains full JSON-LD
    def find_record_sets(obj):
        found = []
        if isinstance(obj, dict):
            if obj.get('@type') in ['RecordSet', 'cr:RecordSet'] or obj.get('@type') == 'cr:RecordSet':
                found.append(obj)
            for v in obj.values():
                found.extend(find_record_sets(v))
        elif isinstance(obj, list):
            for v in obj:
                found.extend(find_record_sets(v))
        return found
    record_set_objs = find_record_sets(croissant_json)
    record_set_dict = {r['@id']: r for r in record_set_objs}
else:
    record_set_dict = {r['@id']: r for r in dataset.record_sets}

print("Found the following RecordSets (by @id):")
for rsid in record_set_dict:
    name = record_set_dict[rsid].get('name', '(no name)')
    print(f"- {rsid}: {name}")
    # List their fields
    fields = record_set_dict[rsid].get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    elif isinstance(fields, list):
        fields = fields
    else:
        fields = []
    print("    Fields:")
    for fld in fields:
        # If using compact form of Croissant, fields may be just strings
        if isinstance(fld, str):
            print(f"      - {fld}")
        elif isinstance(fld, dict):
            print(f"      - {fld.get('@id', str(fld))}")
        else:
            print(f"      - {fld}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. If multiple record sets are present, all will be loaded.

In [ ]:
# Prepare list of available record set @ids
record_sets = list(record_set_dict.keys())
print("Preparing to load record sets:", record_sets)

dataframes = {}
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set] = df
            print(f"Loaded {len(df)} records for RecordSet '{record_set}'.")
        else:
            print(f"No records found for RecordSet '{record_set}'.")
    except Exception as e:
        print(f"Error loading records for '{record_set}':", e)

# Show columns in the first loaded record set (if any)
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"Columns in RecordSet '{first_rs}':", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No data could be loaded into dataframes.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
Operations shown: filtering, normalization, and grouping. **All field references use their Croissant `@id` identifier where possible.**

In [ ]:
# You may need to adjust these values based on the fields printed above
if dataframes:
    # Use the first RecordSet for demonstration
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]

    # Attempt to select a numeric field by inspecting dtypes
    num_cols = df.select_dtypes(include=['number']).columns
    if len(num_cols) == 0:
        # Try to infer numerics from columns with likely numeric names (e.g. ending with '_count', 'MW', 'coverage', etc.)
        cand_cols = [c for c in df.columns if any(s in c.lower() for s in ['count','mw','coverage','abundance','pi'])]
        if cand_cols:
            # Try to coerce first candidate column to numeric
            df[cand_cols[0]] = pd.to_numeric(df[cand_cols[0]], errors='coerce')
            numeric_field = cand_cols[0]
        else:
            print("No obvious numeric columns found for demonstration.")
            numeric_field = None
    else:
        numeric_field = num_cols[0]

    if numeric_field:
        print(f"Using numeric field '{numeric_field}' for demonstration.")
        # Apply a basic filter
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field]).all() else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (mean): {len(filtered_df)} rows.")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to find a grouping field (likely 'accession', 'description', or sample/condition field)
        group_field = None
        for cand in ['accession', 'description','gene','sample','condition']:
            if cand in df.columns:
                group_field = cand
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable group field detected.")
    else:
        print("No numeric field available for EDA.")
else:
    print("No data available for analysis.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Field and RecordSet references are via their `@id`s as per the Croissant schema, where possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and ('numeric_field' in locals()) and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}' in RecordSet '{rs_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # Example scatter plot if a second numeric field exists
    other_num_cols = [c for c in df.columns if c != numeric_field and pd.api.types.is_numeric_dtype(df[c])]
    if other_num_cols:
        plt.figure(figsize=(7,7))
        sns.scatterplot(x=df[numeric_field], y=df[other_num_cols[0]])
        plt.title(f"Scatter plot of '{numeric_field}' vs '{other_num_cols[0]}'")
        plt.xlabel(numeric_field)
        plt.ylabel(other_num_cols[0])
        plt.show()
else:
    print("Not enough numeric fields for visualization.")


## 6. Conclusion
In this notebook, we've loaded the FAIR² Croissant schema describing the human mast cell mass spectrometry dataset and explored its structure and records using the `mlcroissant` library. We demonstrated programmatic filtering, normalization, grouping by fields (with all identifiers referenced via their Croissant `@id` when possible), and visualized distributions of numeric fields.

For more detailed analysis, users may want to further inspect the schema to map additional field `@id`s to dataset columns, tailor filtering criteria, and create advanced visualizations leveraging this harmonized FAIR metadata structure.